In [ ]:
%%writefile flash_15.cu
#include<iostream>
#include<cuda_runtime.h>
using namespace std;

#define TILE_SIZE 32
#define D 64



__global__ void __launch_bounds__(256, 3)
flashAttention_v15(float* __restrict__ Q,
                   float* __restrict__ K,
                   float* __restrict__ V,
                   float* __restrict__ O,
                   int N)
{
    const int warp_id         = threadIdx.x >> 5;
    const int lane_id         = threadIdx.x & 31;
    const int warps_per_block = blockDim.x >> 5;

    const int i = blockIdx.x * warps_per_block + warp_id;
    if (i >= N) return;

    // 32x64x4 = 8KB each
    // total = 16KB shared memory
    __shared__ float K_tile[TILE_SIZE][D];
    __shared__ float V_tile[TILE_SIZE][D];

    const float scale = rsqrtf((float)D);
    const float LOG2E = 1.4426950408f;

    // each lane owns 2 dimensions
    const float qi0 = Q[i * D + lane_id * 2    ] * scale;
    const float qi1 = Q[i * D + lane_id * 2 + 1] * scale;

    float oi0 = 0.f;
    float oi1 = 0.f;

    float m = -INFINITY;
    float l = 0.f;

    for (int tile_start = 0; tile_start < N; tile_start += TILE_SIZE)
    {
        // vectorized cooperative load
        for (int idx = threadIdx.x;
             idx < (TILE_SIZE * D) / 4;
             idx += blockDim.x)
        {
            const int vec_idx = idx;

            const int row = vec_idx >> 4;          // /16
            const int col = (vec_idx & 15) << 2;  // *4

            if (tile_start + row < N)
            {
                reinterpret_cast<float4*>(&K_tile[row][col])[0] =
                    reinterpret_cast<const float4*>(
                        &K[(tile_start + row) * D + col])[0];

                reinterpret_cast<float4*>(&V_tile[row][col])[0] =
                    reinterpret_cast<const float4*>(
                        &V[(tile_start + row) * D + col])[0];
            }
        }

        __syncthreads();

        const int valid_j = min(TILE_SIZE, N - tile_start);
        const int full_j  = (valid_j >> 2) << 2;

        // process 4 rows at a time
        for (int j = 0; j < full_j; j += 4)
        {
            // rolling register fragments
            float2 k0 =
                reinterpret_cast<float2*>(&K_tile[j][lane_id * 2])[0];
            float2 k1 =
                reinterpret_cast<float2*>(&K_tile[j + 1][lane_id * 2])[0];
            float2 k2 =
                reinterpret_cast<float2*>(&K_tile[j + 2][lane_id * 2])[0];
            float2 k3 =
                reinterpret_cast<float2*>(&K_tile[j + 3][lane_id * 2])[0];

            float2 v0 =
                reinterpret_cast<float2*>(&V_tile[j][lane_id * 2])[0];
            float2 v1 =
                reinterpret_cast<float2*>(&V_tile[j + 1][lane_id * 2])[0];
            float2 v2 =
                reinterpret_cast<float2*>(&V_tile[j + 2][lane_id * 2])[0];
            float2 v3 =
                reinterpret_cast<float2*>(&V_tile[j + 3][lane_id * 2])[0];

            // partial dot products
            float s0 = qi0 * k0.x + qi1 * k0.y;
            float s1 = qi0 * k1.x + qi1 * k1.y;
            float s2 = qi0 * k2.x + qi1 * k2.y;
            float s3 = qi0 * k3.x + qi1 * k3.y;

            // packed shuffle reduction
            union Pack64 {
                uint64_t u64;
                struct { float a, b; };
            };

            Pack64 p01, p23;

            p01.a = s0;
            p01.b = s1;

            p23.a = s2;
            p23.b = s3;

            #pragma unroll
            for (int offset = 16; offset > 0; offset >>= 1)
            {
                Pack64 tmp01, tmp23;

                tmp01.u64 =
                    __shfl_down_sync(0xffffffff, p01.u64, offset);

                tmp23.u64 =
                    __shfl_down_sync(0xffffffff, p23.u64, offset);

                p01.a += tmp01.a;
                p01.b += tmp01.b;

                p23.a += tmp23.a;
                p23.b += tmp23.b;
            }

            p01.u64 = __shfl_sync(0xffffffff, p01.u64, 0);
            p23.u64 = __shfl_sync(0xffffffff, p23.u64, 0);

            const float rs0 = p01.a;
            const float rs1 = p01.b;
            const float rs2 = p23.a;
            const float rs3 = p23.b;

            // online softmax
            const float m_old     = m;
            const float local_max =
                fmaxf(fmaxf(rs0, rs1), fmaxf(rs2, rs3));

            const float m_new   = fmaxf(m_old, local_max);
            const float p_scale =
                exp2f((m_old - m_new) * LOG2E);

            const float ep0 =
                exp2f((rs0 - m_new) * LOG2E);

            const float ep1 =
                exp2f((rs1 - m_new) * LOG2E);

            const float ep2 =
                exp2f((rs2 - m_new) * LOG2E);

            const float ep3 =
                exp2f((rs3 - m_new) * LOG2E);

            oi0 = oi0 * p_scale
                + ep0 * v0.x
                + ep1 * v1.x
                + ep2 * v2.x
                + ep3 * v3.x;

            oi1 = oi1 * p_scale
                + ep0 * v0.y
                + ep1 * v1.y
                + ep2 * v2.y
                + ep3 * v3.y;

            l = l * p_scale + ep0 + ep1 + ep2 + ep3;

            m = m_new;
        }

        // tail handling
        for (int j = full_j; j < valid_j; j++)
        {
            float2 k =
                reinterpret_cast<float2*>(&K_tile[j][lane_id * 2])[0];

            float2 v =
                reinterpret_cast<float2*>(&V_tile[j][lane_id * 2])[0];

            float s = qi0 * k.x + qi1 * k.y;

            #pragma unroll
            for (int offset = 16; offset > 0; offset >>= 1)
                s += __shfl_down_sync(0xffffffff, s, offset);

            s = __shfl_sync(0xffffffff, s, 0);

            const float m_old   = m;
            const float m_new   = fmaxf(m_old, s);

            const float p_scale =
                exp2f((m_old - m_new) * LOG2E);

            const float ep =
                exp2f((s - m_new) * LOG2E);

            oi0 = oi0 * p_scale + ep * v.x;
            oi1 = oi1 * p_scale + ep * v.y;

            l = l * p_scale + ep;

            m = m_new;
        }

        __syncthreads();
    }

    const float inv_l = __fdividef(1.f, l);

    O[i * D + lane_id * 2    ] = oi0 * inv_l;
    O[i * D + lane_id * 2 + 1] = oi1 * inv_l;
}



#define CUDA_CHECK(call)                                      \
{                                                             \
    cudaError_t err = call;                                   \
    if (err != cudaSuccess) {                                 \
        cout << "CUDA error: " << cudaGetErrorString(err)     \
             << " at " << __FILE__ << ":" << __LINE__ << endl;\
        exit(1);                                              \
    }                                                         \
}


int main(){
    int N = 8192;
    int d = 64;

    size_t size = N * d * sizeof(float);

    float *h_Q, *h_K, *h_V, *h_O;
    float *d_Q, *d_K, *d_V, *d_O;

    h_Q = (float*)malloc(size);
    h_K = (float*)malloc(size);
    h_V = (float*)malloc(size);
    h_O = (float*)malloc(size);

    for(int i = 0; i < N * d; i++){
        h_Q[i] = static_cast<float>(rand()) / RAND_MAX;
        h_K[i] = static_cast<float>(rand()) / RAND_MAX;
        h_V[i] = static_cast<float>(rand()) / RAND_MAX;
    }

    CUDA_CHECK(cudaMalloc((void**)&d_Q, size));
    CUDA_CHECK(cudaMalloc((void**)&d_K, size));
    CUDA_CHECK(cudaMalloc((void**)&d_V, size));
    CUDA_CHECK(cudaMalloc((void**)&d_O, size));

    CUDA_CHECK(cudaMemcpy(d_Q, h_Q, size, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_K, h_K, size, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_V, h_V, size, cudaMemcpyHostToDevice));

    // ── kernel config ──────────────────────────────────────────
    // Each warp handles 2 rows → need N/2 warps total
    int threads          = 256;
    int warps_per_block  = threads / 32;
    int total_warps      = (N / 2 + 1);           // warps needed
    int blocks           = (total_warps + warps_per_block - 1) / warps_per_block;

    // warmup
    for(int i = 0; i < 100; i++)
        flashAttention_v15<<<blocks, threads>>>(d_Q, d_K, d_V, d_O, N);
    cudaDeviceSynchronize();

    // timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    cudaEventRecord(start);

    const int TIMED_RUNS = 100;
    for(int i = 0; i < TIMED_RUNS; i++)
        flashAttention_v15<<<blocks, threads>>>(d_Q, d_K, d_V, d_O, N);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms = 0;
    cudaEventElapsedTime(&ms, start, stop);
    ms /= TIMED_RUNS;

    double flops     = 4.0 * N * N * d;
    double bytes     = 3.0 * N * N * sizeof(float);
    double gflops    = (flops / (ms / 1000.0)) / 1e9;
    double bandwidth = (bytes / (ms / 1000.0)) / 1e9;
    double intensity = flops / bytes;

    cout << "Time: "                      << ms        << " ms\n";
    cout << "GFLOPS: "                    << gflops    << "\n";
    cout << "Memory Throughput (GB/s): "  << bandwidth << "\n";
    cout << "Arithmetic Intensity: "      << intensity << "\n";

    printf("METRIC,time_ms,%.4f\n",  ms);
    printf("METRIC,gflops,%.2f\n",   gflops);
    printf("METRIC,bandwidth,%.2f\n",bandwidth);
    printf("METRIC,intensity,%.2f\n", intensity);

    cudaFree(d_Q); cudaFree(d_K); cudaFree(d_V); cudaFree(d_O);
    free(h_Q);     free(h_K);     free(h_V);     free(h_O);

    return 0;
}

Overwriting flash_15.cu
